# Data 620 Week 8 Assignment - Project 3
### Author: Pedro P Gatica Jr

Project 3 Rubric:
1. Using any of the three classifiers described in chapter 6 of Natural Language Processing with Python, and any features you can think of, build the best name gender classifier you can.
2. Begin by splitting the Names Corpus into three subsets: 500 words for the test set, 500 words for the dev-test set, and the remaining 6900 words for the training set.
3. Then, starting with the example name gender classifier, make incremental improvements. Use the dev-test set to check your progress. Once you are satisfied with your classifier, check its final performance on the test set.
4. How does the performance on the test set compare to the performance on the dev-test set? Is this what you'd expect?

In [1]:
import pandas as pd
import csv
import os
from datetime import datetime
import random
import nltk
from nltk.corpus import names
from collections import Counter

nltk.download('names')

[nltk_data] Downloading package names to
[nltk_data]     /Users/Audiorunner13/nltk_data...
[nltk_data]   Package names is already up-to-date!


True

In [2]:
RESULTS_FILE = "project3_results.csv"
results = []   # in-memory log of every experiment

In [3]:
labeled_names = ([(name, 'male') for name in names.words('male.txt')] +
                 [(name, 'female') for name in names.words('female.txt')])

##### This code is splitting a big list of names into three separate piles, which is a standard first step when building a machine learning model.
##### The three slices
##### test_names = labeled_names[:500] takes the first 500 names.
##### devtest_names = labeled_names[500:1000] takes the next 500 (positions 500 to 999).
##### train_names = labeled_names[1000:] takes everything else, from position 1000 to the end.

In [4]:
random.seed(42)
random.shuffle(labeled_names)

test_names    = labeled_names[:500]
devtest_names = labeled_names[500:1000]
train_names   = labeled_names[1000:]

print(len(train_names), len(devtest_names), len(test_names))

6944 500 500


##### I have decided to use the Naive Bayes classifier. It is called Naive because it makes a simplifying assumption: it treats every clue as independent of the others. In reality, clues are often connected.  It’s simple, fast, and works surprisingly well in practice.
##### It’s named after Thomas Bayes, an 18th-century mathematician. His rule is to update your belief when you see new evidence. You start with a guess, see a clue, and adjust.

In [5]:
def run_experiment(version, description, feature_fn):
    """Train on train_names, evaluate on dev-test, and log the result."""
    train_set   = [(feature_fn(n), g) for (n, g) in train_names]
    devtest_set = [(feature_fn(n), g) for (n, g) in devtest_names]

    clf = nltk.NaiveBayesClassifier.train(train_set)
    acc = nltk.classify.accuracy(clf, devtest_set)

    # Compare against the previous version, if there is one
    change = acc - results[-1]["devtest_accuracy"] if results else 0.0

    results.append({
        "version": version,
        "description": description,
        "devtest_accuracy": round(acc, 4),
        "change_from_prev": round(change, 4),
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    })
    save_results()

    print(f"{version}: {acc:.4f} ({change:+.4f})  {description}")
    return clf

In [6]:
def save_results():
    """Overwrite the CSV with the full results log."""
    with open(RESULTS_FILE, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)

##### A function that turns a name into a “clue sheet.” The classifier can’t read names directly. It needs labeled clues, so this function boils each name down to one clue. Here, that clue is the second-to-last letter.

In [7]:
# Baseline feature extractor (from the chapter)
def gender_features(name):
    return {'last_2_letters': name[-2]}

##### These loop through every name and convert it into a pair: (clue sheet, correct answer). For “Omar,” the pair would be ({'last_2_letters': 'a'}, 'male'). The training pairs are what the classifier studies, and the dev-test pairs are the practice exam.

In [8]:
# Convert each (name, gender) pair into a (features, gender) pair
train_set   = [(gender_features(n), g) for (n, g) in train_names]
devtest_set = [(gender_features(n), g) for (n, g) in devtest_names]

In [9]:
pd.DataFrame(train_names[:20], columns=["name", "gender"])

,name,gender
0,Babette,female
1,Omar,male
2,Clemmie,female
3,Constancy,female
4,Emil,male
5,Jon,male
6,Essy,female
7,Theda,female
8,Malinda,female
9,Charlie,male


In [10]:
# Train the Naive Bayes classifier
classifier = nltk.NaiveBayesClassifier.train(train_set)

In [11]:
# Evaluate on the dev-test set (NOT the test set)
print("Dev-test accuracy:", nltk.classify.accuracy(classifier, devtest_set))

Dev-test accuracy: 0.672


In [12]:
# Look at the most informative features
classifier.show_most_informative_features(10)

Most Informative Features
          last_2_letters = 'o'              male : female =      7.7 : 1.0
          last_2_letters = 'u'              male : female =      6.5 : 1.0
          last_2_letters = 'j'            female : male   =      4.1 : 1.0
          last_2_letters = 'E'              male : female =      4.0 : 1.0
          last_2_letters = 'y'            female : male   =      3.6 : 1.0
          last_2_letters = 'f'              male : female =      3.3 : 1.0
          last_2_letters = 'n'            female : male   =      3.0 : 1.0
          last_2_letters = 'a'              male : female =      2.6 : 1.0
          last_2_letters = 't'            female : male   =      2.6 : 1.0
          last_2_letters = 's'            female : male   =      2.6 : 1.0


##### Above we list the 10 strongest clues, ranked by how lopsided they are.
##### The way we read each line is:
##### Take the first row:
##### last_2_letters = 'o'     male : female = 7.7 : 1.0
##### When this clue shows up, the name is about 7.7 times more likely to be male than female.
##### The ratio compares the two sides. The bigger the first number, the stronger the clue, and the label named first is the one it points to.

In [13]:
for name in ['Neo', 'Trinity', 'Morpheus', 'Alexandra']:
    print(name, '->', classifier.classify(gender_features(name)))

Neo -> female
Trinity -> female
Morpheus -> male
Alexandra -> female


##### Next 

In [14]:
# v1: baseline from the chapter
def features_v1(name):
    return {'last_letter': name[-1].lower()}

In [15]:
# v2: add first letter
def features_v2(name):
    f = features_v1(name)
    f['first_letter'] = name[0].lower()
    return f

In [16]:
# v3: add last two letters
def features_v3(name):
    f = features_v2(name)
    f['suffix2'] = name[-2:].lower()
    return f

In [17]:
# v4: add last three letters and length
def features_v4(name):
    f = features_v3(name)
    f['suffix3'] = name[-3:].lower()
    f['length'] = len(name)
    return f

In [18]:
# v5: add letter-presence features
def features_v5(name):
    f = features_v4(name)
    for letter in 'abcdefghijklmnopqrstuvwxyz':
        f[f'has({letter})'] = (letter in name.lower())
    return f

In [19]:
def run_experiment_clf(version, description, feature_fn, trainer):
    """Like run_experiment, but takes any NLTK trainer and logs to the same CSV."""
    train_set   = [(feature_fn(n), g) for (n, g) in train_names]
    devtest_set = [(feature_fn(n), g) for (n, g) in devtest_names]

    clf = trainer(train_set)
    acc = nltk.classify.accuracy(clf, devtest_set)
    change = acc - results[-1]["devtest_accuracy"] if results else 0.0

    results.append({
        "version": version,
        "description": description,
        "devtest_accuracy": round(acc, 4),
        "change_from_prev": round(change, 4),
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    })
    save_results()

    print(f"{version}: {acc:.4f} ({change:+.4f})  {description}")
    return clf

In [20]:
run_experiment("v1", "last letter (baseline)",         features_v1)

v1: 0.7540 (+0.0000)  last letter (baseline)


In [21]:
run_experiment("v2", "+ first letter",                 features_v2)

v2: 0.7540 (+0.0000)  + first letter


In [22]:
run_experiment("v3", "+ last two letters",             features_v3)

v3: 0.7820 (+0.0280)  + last two letters


In [23]:
run_experiment("v4", "+ last three letters, length",   features_v4)

v4: 0.7920 (+0.0100)  + last three letters, length


In [24]:
run_experiment("v5", "+ letter presence (a-z)",        features_v5)

v5: 0.8060 (+0.0140)  + letter presence (a-z)


In [25]:
df = pd.read_csv(RESULTS_FILE)
print(df.to_string(index=False))

version                  description  devtest_accuracy  change_from_prev           timestamp
     v1       last letter (baseline)             0.754             0.000 2026-11-06 12:28:16
     v2               + first letter             0.754             0.000 2026-11-06 12:28:16
     v3           + last two letters             0.782             0.028 2026-11-06 12:28:16
     v4 + last three letters, length             0.792             0.010 2026-11-06 12:28:16
     v5      + letter presence (a-z)             0.806             0.014 2026-11-06 12:28:16


##### This table is a progress report on building a name-guesser. Each row is a new version of the classifier where I gave it more clues to work with, and the accuracy column shows how often it correctly guessed male vs. female on the 500 dev-test names.
##### v1 (baseline): With only the last letter to go on, the model gets about 3 out of 4 names right. This is the benchmark to beat. (It’s a decent start because last letters carry a lot of signal, like “a” often being female.)
##### v2 (first letter): Adding the first letter changed nothing. The first letter of a name just isn’t very telling. A name starting with “M” or “J” could easily be either gender, so the model gained no new useful information.
##### v3 (last two letters): This was the biggest improvement. Endings like “-ie,” “-yn,” or “-on” are much more revealing than a single letter, so looking at two letters gave the model sharper clues.
##### v4 (last three letters + length): Another small gain. Endings like “-ine” or “-ton” help a bit more, and name length adds a little extra hint. The improvements are shrinking, though.
##### v5 (letters present anywhere): Checking which letters appear in the name at all pushed accuracy to its best so far, 80.6%.

In [26]:
best_features = features_v5   # change to whichever version won

In [27]:
train_set = [(best_features(n), g) for (n, g) in train_names]
test_set  = [(best_features(n), g) for (n, g) in test_names]
final_clf = nltk.NaiveBayesClassifier.train(train_set)

In [28]:
test_acc = nltk.classify.accuracy(final_clf, test_set)
best_devtest = max(r["devtest_accuracy"] for r in results)

In [29]:
print(f"Dev-test accuracy: {best_devtest:.4f}")
print(f"Test accuracy:     {test_acc:.4f}")
print(f"Difference:        {test_acc - best_devtest:+.4f}")

Dev-test accuracy: 0.8060
Test accuracy:     0.7840
Difference:        -0.0220


In [30]:
# Save the final result alongside the log
with open("project3_final.txt", "w") as f:
    f.write(f"Best dev-test: {best_devtest:.4f}\nTest: {test_acc:.4f}\n")

In [31]:
# Retrain the v5 classifier
train_set = [(features_v5(n), g) for (n, g) in train_names]
clf_v5 = nltk.NaiveBayesClassifier.train(train_set)

In [33]:
# Collect dev-test errors
errors = []
for (name, tag) in devtest_names:
    guess = clf_v5.classify(features_v5(name))
    if guess != tag:
        errors.append((tag, guess, name))

print(f"Errors: {len(errors)} of {len(devtest_names)}")

Errors: 97 of 500


##### This was the final exam. After building five versions of the classifier and picking the best one, I tested it once on the 500 names it has never seen. The model correctly guessed about 78 out of every 100 names, which is 392 of the 500.
##### Why is the final score lower? It’s completely normal. There are two reasons:
##### Mild “peeking.” I tried five versions and kept the one that scored best on the dev-test names. That’s like picking the study method that happened to do best on a practice test. Some of that success was partly luck, and luck doesn’t carry over to a fresh exam.
##### Small sample size. With only 500 names, each name is worth 0.2 points. A 2.2-point gap is only about 11 names, which is within normal random variation.

##### A gap of about 2 points means the model learned real patterns rather than just memorizing the training names. If it had dropped to 65% or 70%, that would suggest “overfitting” (memorizing rather than learning). Here the two scores are close, so the dev-test results were a fairly honest preview.
##### If I showed this model brand-new names, I would expect it to get roughly 3 to 4 out of every 5 right, with some wobble. That’s a solid improvement over the 75.4% baseline, though note the baseline was measured on the dev-test set, so the true improvement is probably a bit smaller than the 5 points it appeared to be.

### Confusion matrix: which direction are the mistakes?

In [34]:
confusion = Counter((tag, guess) for (tag, guess, name) in errors)
print("male   -> predicted female:", confusion[('male', 'female')])
print("female -> predicted male:  ", confusion[('female', 'male')])

male   -> predicted female: 51
female -> predicted male:   46


##### The model makes about the same number of mistakes in each direction, so it isn’t favoring one gender. The errors aren’t lopsided, which means improving the model will probably require better clues overall rather than fixing one specific blind spot.
##### A good next step is to look at the actual names behind those 97 mistakes (for example, errors[:20]). I will likely spot patterns, such as unisex names like “Jordan” or “Taylor,” which are hard for any model to get right.

### Which endings cause the most errors?

In [35]:
print("\nMost common last letters among errors:")
for letter, count in Counter(n[-1].lower() for (_, _, n) in errors).most_common(8):
    print(f"  -{letter}: {count}")


Most common last letters among errors:
  -e: 25
  -y: 19
  -n: 13
  -l: 9
  -t: 6
  -s: 5
  -d: 4
  -i: 4


In [36]:
print("\nMost common last-two-letter endings among errors:")
for suf, count in Counter(n[-2:].lower() for (_, _, n) in errors).most_common(10):
    print(f"  -{suf}: {count}")


Most common last-two-letter endings among errors:
  -ie: 9
  -ny: 6
  -en: 5
  -ne: 4
  -in: 4
  -ll: 3
  -dy: 3
  -ey: 2
  -lo: 2
  -rd: 2


### Compare error rate per ending against how common the ending is. 
##### An ending with many errors may simply be frequent, so this normalizes it:

In [38]:
suffix_total = Counter(n[-2:].lower() for (n, _) in devtest_names)
suffix_err   = Counter(n[-2:].lower() for (_, _, n) in errors)

print("\nSuffixes with the highest error rate (at least 5 names):")

rows = [(s, suffix_err[s], suffix_total[s], suffix_err[s] / suffix_total[s])
        for s in suffix_total if suffix_total[s] >= 5]
for s, e, t, r in sorted(rows, key=lambda x: -x[3])[:10]:
    print(f"  -{s}: {e}/{t} wrong ({r:.0%})")


Suffixes with the highest error rate (at least 5 names):
  -ny: 6/8 wrong (75%)
  -ll: 3/5 wrong (60%)
  -en: 5/11 wrong (45%)
  -is: 2/6 wrong (33%)
  -ie: 9/28 wrong (32%)
  -rd: 2/7 wrong (29%)
  -el: 2/7 wrong (29%)
  -ey: 2/8 wrong (25%)
  -in: 4/16 wrong (25%)
  -th: 2/9 wrong (22%)


### Read the actual mistakes. 
##### This is the step that tends to give feature ideas:

In [39]:
print("\nMale names predicted female:")
print(sorted(n for (t, g, n) in errors if t == 'male'))

print("\nFemale names predicted male:")
print(sorted(n for (t, g, n) in errors if t == 'female'))


Male names predicted female:
['Abby', 'Aldis', 'Allen', 'Alley', 'Alphonse', 'Andre', 'Antony', 'Bailie', 'Barri', 'Benjamen', 'Benji', 'Benny', 'Boyce', 'Brandy', 'Claire', 'Costa', 'Daryl', 'Davide', 'Erl', 'Etienne', 'Georgie', 'Germaine', 'Jessee', 'Kenny', 'Kermie', 'Kingsly', 'Manny', 'Marlowe', 'Martainn', 'Melvyn', 'Meredith', 'Merlin', 'Merry', 'Nathaniel', 'Normie', 'Ole', 'Oral', 'Richie', 'Roddie', 'Rodrique', 'Seth', 'Sollie', 'Tabby', 'Tanny', 'Thaine', 'Tuckie', 'Tye', 'Vinny', 'Wayne', 'Yehudi', 'Zacharie']

Female names predicted male:
['Beatriz', 'Bliss', 'Brigid', 'Brooks', 'Cherey', 'Cordy', 'Dallas', 'Doralin', 'Dory', 'Eden', 'Eleanor', 'Gillan', 'Ginnifer', 'Glad', 'Gretchen', 'Grissel', 'Harriott', 'Hildagard', 'Ivett', 'Ivy', 'Joell', 'Lil', 'Lindsay', 'Lust', 'Margalo', 'Marlo', 'Mei', 'Melicent', 'Mellicent', 'Midge', 'Moll', 'Pris', 'Raf', 'Raynell', 'Robbin', 'Roselin', 'Shannen', 'Sharon', 'Sher', 'Stoddard', 'Tamar', 'Trish', 'Trudy', 'Ulrike', 'Vikky', 

In [40]:
totals = Counter(tag for (_, tag) in devtest_names)
print("Dev-test class counts:", dict(totals))
print(f"Male error rate:   {51 / totals['male']:.1%}")
print(f"Female error rate: {46 / totals['female']:.1%}")

Dev-test class counts: {'female': 297, 'male': 203}
Male error rate:   25.1%
Female error rate: 15.5%


### Conclusion
##### The goal was to build a simple program that guesses whether a first name is male or female. We used about 7,900 labeled names, shuffled them, and split them into three groups: most for teaching the program (training), 500 for practice checks while building it (dev-test), and 500 locked away for a final, honest exam (test). The program was a Naive Bayes classifier, which makes its guess by weighing small clues about each name.
##### More useful clues made the program smarter. We built five versions, each with more clues, and practice accuracy climbed from 75.4% to 80.6%. The biggest jump came from looking at the endings of names rather than a single letter. Some clues, like the first letter, added nothing. Each new clue helped a little less than the one before it.
##### The final exam score was 78.4%. On 500 names it had never seen, the program was right about 78 times out of 100. That is about 2 points lower than its practice score, which is normal. Picking the best of several versions on the same practice set gives a slightly flattering result, and 500 names is a small sample. The small gap suggests the program learned real patterns rather than memorizing names.
##### Its mistakes were balanced. It wrongly called male names female 51 times and female names male 46 times, so it doesn’t strongly favor either gender.

#### YouTube Video Link

https://youtu.be/DHuOWh3qcV8